# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [ ]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 🔹 Model

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [ ]:
model.ALIMENTS = Set(initialize=['Brownie', 'Cremeglacee', 'Cola', 'Gateau'])
model.INGREDIENTS = Set(initialize=['Calories', 'Chocolat', 'Sucre', 'Matieregrasse'])
model.ARCS = Set(dimen=2, initialize=[(i0,i1) for i0 in model.ALIMENTS for i1 in model.INGREDIENTS])

## 🔹 Parameters

In [ ]:
model.Prix = Param(model.ALIMENTS, initialize={'Brownie': 50.0, 'Cremeglacee': 20.0, 'Cola': 30.0, 'Gateau': 80.0}, within=NonNegativeReals)
model.Calories = Param(model.ALIMENTS, initialize={'Brownie': 400.0, 'Cremeglacee': 200.0, 'Cola': 150.0, 'Gateau': 500.0}, within=NonNegativeReals)
model.DIETEJOUR = Param(model.INGREDIENTS, initialize={'Calories': 500.0, 'Chocolat': 6.0, 'Sucre': 10.0, 'Matieregrasse': 8.0}, within=NonNegativeReals)
model.QTEING = Param(model.ALIMENTS, model.INGREDIENTS, initialize={('Brownie', 'Calories'): 400.0, ('Brownie', 'Chocolat'): 3.0, ('Brownie', 'Sucre'): 2.0, ('Brownie', 'Matieregrasse'): 2.0, ('Cremeglacee', 'Calories'): 200.0, ('Cremeglacee', 'Chocolat'): 2.0, ('Cremeglacee', 'Sucre'): 2.0, ('Cremeglacee', 'Matieregrasse'): 4.0, ('Cola', 'Calories'): 150.0, ('Cola', 'Chocolat'): 0.0, ('Cola', 'Sucre'): 4.0, ('Cola', 'Matieregrasse'): 1.0, ('Gateau', 'Calories'): 500.0, ('Gateau', 'Chocolat'): 0.0, ('Gateau', 'Sucre'): 4.0, ('Gateau', 'Matieregrasse'): 5.0}, within=NonNegativeReals)

## 🔹 Variables

In [ ]:
model.X = Var(model.ALIMENTS, domain=NonNegativeReals)

## 🔹 Constraints

In [ ]:
model.c0 = Constraint(expr=sum(model.Calories[a]*model.X[a] for a in model.ALIMENTS) >= 500)
model.c_for_0 = ConstraintList()
for i in model.INGREDIENTS:
    model.c_for_0.add(sum(model.QTEING[a, i] * model.X[a] for a in model.ALIMENTS) >= model.DIETEJOUR[i])

## 🔹 Objective

In [ ]:
model.obj = Objective(expr=sum(model.Prix[a] * model.X[a] for a in model.ALIMENTS), sense=minimize)

## ⚙️ Résolution du modèle

In [ ]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

## 🎯 Valeur de la fonction objective

In [ ]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

## 📊 Valeurs optimales des variables

In [ ]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')